In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Locate project root
# --------------------------------------------------

current_path = Path.cwd()

project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "src" / "config.py").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing src/config.py."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")


# --------------------------------------------------
# 2. Import existing project functions
# --------------------------------------------------

from src.data_loader import load_raw_data
from src.features.metric_definitions import add_metric_features


# --------------------------------------------------
# 3. Load delivery-level data
# --------------------------------------------------

df = load_raw_data()
df_features = add_metric_features(df)

print(f"Raw dataset shape: {df.shape}")
print(f"Feature dataset shape: {df_features.shape}")

display(df_features.head())

Project root: /Users/soham/MyDocs/Projects/Cricket Analysis/Hackathon - DPL 2026/CodeFiles/Hackathon_DPL2026
Raw dataset shape: (34967, 61)
Feature dataset shape: (34967, 71)


,row_id,p_match,inns,bat,p_bat,team_bat,bowl,p_bowl,team_bowl,ball,...,legal_delivery,batting_dot,bowling_dot,bowler_credited_wicket,match_phase,boundary_ball,boundary_runs,wides,noball_extras,wide_noball_extras
0,225793,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,1,...,True,True,True,False,Powerplay,False,0,0,0,0
1,225794,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,2,...,True,True,False,False,Powerplay,False,0,0,0,0
2,225795,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,3,...,True,True,True,False,Powerplay,False,0,0,0,0
3,225796,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,4,...,True,False,False,False,Powerplay,False,0,0,0,0
4,225797,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,5,...,True,True,True,False,Powerplay,False,0,0,0,0


In [2]:
# --------------------------------------------------
# 1. Validate delivery grain
# --------------------------------------------------

delivery_key = ["p_match", "inns", "ball_id"]

missing_key_columns = [
    col for col in delivery_key
    if col not in df_features.columns
]

if missing_key_columns:
    raise KeyError(
        f"Missing delivery-key columns: {missing_key_columns}"
    )

duplicate_delivery_keys = df_features.duplicated(
    subset=delivery_key,
    keep=False,
)

print("DELIVERY GRAIN VALIDATION")
print(f"Total delivery records: {len(df_features):,}")
print(
    "Distinct delivery keys: "
    f"{df_features[delivery_key].drop_duplicates().shape[0]:,}"
)
print(
    "Rows involved in duplicate delivery keys: "
    f"{duplicate_delivery_keys.sum():,}"
)


# --------------------------------------------------
# 2. Validate required match-up dimensions
# --------------------------------------------------

required_columns = [
    "p_match",
    "year",
    "inns",
    "ball_id",
    "team_bat",
    "team_bowl",
    "p_bat",
    "bat",
    "p_bowl",
    "bowl",
    "over_num",
    "batruns",
    "score",
    "out",
    "dismissal",
    "wide",
    "noball",
    "match_phase",
]

missing_columns = [
    col for col in required_columns
    if col not in df_features.columns
]

print("\nMATCH-UP COLUMN VALIDATION")
print(f"Required columns: {len(required_columns)}")
print(f"Missing columns: {missing_columns}")


# --------------------------------------------------
# 3. Validate player identity consistency
# --------------------------------------------------

batter_identity = (
    df_features[["p_bat", "bat"]]
    .dropna()
    .drop_duplicates()
)

bowler_identity = (
    df_features[["p_bowl", "bowl"]]
    .dropna()
    .drop_duplicates()
)

batter_identity_conflicts = (
    batter_identity.groupby("p_bat")["bat"]
    .nunique()
    .gt(1)
)

bowler_identity_conflicts = (
    bowler_identity.groupby("p_bowl")["bowl"]
    .nunique()
    .gt(1)
)

print("\nPLAYER IDENTITY VALIDATION")
print(
    "Batter IDs mapped to multiple names: "
    f"{batter_identity_conflicts.sum()}"
)
print(
    "Bowler IDs mapped to multiple names: "
    f"{bowler_identity_conflicts.sum()}"
)


# --------------------------------------------------
# 4. Inspect available teams and match-up coverage
# --------------------------------------------------

kkr_batting = df_features.loc[
    df_features["team_bat"].eq("Kolkata Knight Riders")
].copy()

kkr_bowling = df_features.loc[
    df_features["team_bowl"].eq("Kolkata Knight Riders")
].copy()

print("\nKKR MATCH-UP COVERAGE")
print(
    "Distinct matches represented while KKR batted: "
    f"{kkr_batting['p_match'].nunique()}"
)
print(
    "Distinct matches represented while KKR bowled: "
    f"{kkr_bowling['p_match'].nunique()}"
)

print("\nOPPONENTS FACED BY KKR BATTERS")
display(
    kkr_batting.groupby("team_bowl")
    .agg(
        deliveries=("ball_id", "size"),
        matches=("p_match", "nunique"),
        batters=("p_bat", "nunique"),
        bowlers=("p_bowl", "nunique"),
    )
    .sort_values("deliveries", ascending=False)
    .reset_index()
)

print("\nOPPONENTS FACED BY KKR BOWLERS")
display(
    kkr_bowling.groupby("team_bat")
    .agg(
        deliveries=("ball_id", "size"),
        matches=("p_match", "nunique"),
        batters=("p_bat", "nunique"),
        bowlers=("p_bowl", "nunique"),
    )
    .sort_values("deliveries", ascending=False)
    .reset_index()
)

DELIVERY GRAIN VALIDATION
Total delivery records: 34,967
Distinct delivery keys: 34,967
Rows involved in duplicate delivery keys: 0

MATCH-UP COLUMN VALIDATION
Required columns: 18
Missing columns: []

PLAYER IDENTITY VALIDATION
Batter IDs mapped to multiple names: 0
Bowler IDs mapped to multiple names: 0

KKR MATCH-UP COVERAGE
Distinct matches represented while KKR batted: 28
Distinct matches represented while KKR bowled: 28

OPPONENTS FACED BY KKR BATTERS


,team_bowl,deliveries,matches,batters,bowlers
0,Sunrisers Hyderabad,534,5,17,14
1,Royal Challengers Bengaluru,488,4,16,14
2,Chennai Super Kings,375,3,18,10
3,Delhi Capitals,365,3,16,10
4,Lucknow Super Giants,359,3,13,10
5,Punjab Kings,353,3,13,10
6,Mumbai Indians,352,3,14,12
7,Rajasthan Royals,250,2,13,8
8,Gujarat Titans,246,2,10,8



OPPONENTS FACED BY KKR BOWLERS


,team_bat,deliveries,matches,batters,bowlers
0,Sunrisers Hyderabad,618,5,17,11
1,Royal Challengers Bengaluru,476,4,19,12
2,Punjab Kings,365,3,13,13
3,Chennai Super Kings,357,3,10,12
4,Delhi Capitals,357,3,20,11
5,Lucknow Super Giants,357,3,16,9
6,Mumbai Indians,329,3,13,10
7,Gujarat Titans,237,2,7,9
8,Rajasthan Royals,212,2,9,10


In [4]:
# --------------------------------------------------
# 1. Define legal deliveries and bowler-credited dismissals
# --------------------------------------------------

df_features["legal_delivery_matchup"] = (
    df_features["wide"].eq(0)
    & df_features["noball"].eq(0)
)

df_features["bowler_credited_dismissal_matchup"] = (
    df_features["out"].eq(True)
    & df_features["dismissal"].isin(
        [
            "caught",
            "bowled",
            "leg before wicket",
            "stumped",
            "hit wicket",
        ]
    )
)


# --------------------------------------------------
# 2. Create KKR subsets AFTER adding the new columns
# --------------------------------------------------

kkr_batting_deliveries = df_features.loc[
    df_features["team_bat"].eq("Kolkata Knight Riders")
].copy()

kkr_bowling_deliveries = df_features.loc[
    df_features["team_bowl"].eq("Kolkata Knight Riders")
].copy()


# --------------------------------------------------
# 3. Inspect season and phase coverage
# --------------------------------------------------

print("KKR BATTING DELIVERY COUNTS BY SEASON AND PHASE")

display(
    kkr_batting_deliveries.groupby(
        ["year", "match_phase"],
        dropna=False,
    )
    .agg(
        delivery_events=("ball_id", "size"),
        legal_balls=("legal_delivery_matchup", "sum"),
        matches=("p_match", "nunique"),
        batters=("p_bat", "nunique"),
        bowlers=("p_bowl", "nunique"),
    )
    .reset_index()
    .sort_values(["year", "match_phase"])
)

print("KKR BOWLING DELIVERY COUNTS BY SEASON AND PHASE")

display(
    kkr_bowling_deliveries.groupby(
        ["year", "match_phase"],
        dropna=False,
    )
    .agg(
        delivery_events=("ball_id", "size"),
        legal_balls=("legal_delivery_matchup", "sum"),
        matches=("p_match", "nunique"),
        batters=("p_bat", "nunique"),
        bowlers=("p_bowl", "nunique"),
    )
    .reset_index()
    .sort_values(["year", "match_phase"])
)


# --------------------------------------------------
# 4. Validate the subsets
# --------------------------------------------------

print("\nMATCH-UP SUBSET VALIDATION")
print("KKR batting rows:", len(kkr_batting_deliveries))
print("KKR bowling rows:", len(kkr_bowling_deliveries))

print(
    "KKR batting matches:",
    kkr_batting_deliveries["p_match"].nunique(),
)
print(
    "KKR bowling matches:",
    kkr_bowling_deliveries["p_match"].nunique(),
)

print(
    "Missing batting player IDs:",
    kkr_batting_deliveries["p_bat"].isna().sum(),
)
print(
    "Missing bowling player IDs:",
    kkr_bowling_deliveries["p_bowl"].isna().sum(),
)

KKR BATTING DELIVERY COUNTS BY SEASON AND PHASE


,year,match_phase,delivery_events,legal_balls,matches,batters,bowlers
0,2023,Death,418,387,14,13,35
1,2023,Middle,781,756,14,12,53
2,2023,Powerplay,529,504,14,11,38
3,2024,Death,304,269,12,12,32
4,2024,Middle,759,721,14,11,51
5,2024,Powerplay,531,504,14,8,36


KKR BOWLING DELIVERY COUNTS BY SEASON AND PHASE


,year,match_phase,delivery_events,legal_balls,matches,batters,bowlers
0,2023,Death,374,345,13,38,13
1,2023,Middle,758,745,14,53,13
2,2023,Powerplay,525,504,14,35,13
3,2024,Death,328,316,14,42,9
4,2024,Middle,783,756,14,56,9
5,2024,Powerplay,540,504,14,41,7



MATCH-UP SUBSET VALIDATION
KKR batting rows: 3322
KKR bowling rows: 3308
KKR batting matches: 28
KKR bowling matches: 28
Missing batting player IDs: 0
Missing bowling player IDs: 0


In [5]:
# --------------------------------------------------
# 1. Prepare KKR batting matchup records
# --------------------------------------------------

batting_matchup_data = kkr_batting_deliveries.copy()

batting_matchup_data["batter_legal_ball"] = (
    batting_matchup_data["ballfaced"].eq(1)
)

batting_matchup_data["batter_dot_ball"] = (
    batting_matchup_data["batter_legal_ball"]
    & batting_matchup_data["batruns"].eq(0)
)

batting_matchup_data["batter_boundary_ball"] = (
    batting_matchup_data["batter_legal_ball"]
    & batting_matchup_data["batruns"].isin([4, 6])
)

batting_matchup_data["batter_dismissal"] = (
    batting_matchup_data["out"].eq(True)
    & batting_matchup_data["p_out"].eq(
        batting_matchup_data["p_bat"]
    )
)


# --------------------------------------------------
# 2. Aggregate at batter-bowler-season-phase grain
# --------------------------------------------------

batting_matchups = (
    batting_matchup_data.groupby(
        [
            "year",
            "match_phase",
            "p_bat",
            "bat",
            "p_bowl",
            "bowl",
        ],
        dropna=False,
    )
    .agg(
        balls_faced=("batter_legal_ball", "sum"),
        runs_scored=("batruns", "sum"),
        dot_balls=("batter_dot_ball", "sum"),
        boundary_balls=("batter_boundary_ball", "sum"),
        batter_dismissals=("batter_dismissal", "sum"),
        matches=("p_match", "nunique"),
    )
    .reset_index()
)


# --------------------------------------------------
# 3. Calculate matchup metrics
# --------------------------------------------------

batting_matchups["strike_rate"] = (
    100
    * batting_matchups["runs_scored"]
    / batting_matchups["balls_faced"].replace(0, np.nan)
)

batting_matchups["dot_ball_pct"] = (
    100
    * batting_matchups["dot_balls"]
    / batting_matchups["balls_faced"].replace(0, np.nan)
)

batting_matchups["boundary_ball_pct"] = (
    100
    * batting_matchups["boundary_balls"]
    / batting_matchups["balls_faced"].replace(0, np.nan)
)

batting_matchups["dismissal_rate_pct"] = (
    100
    * batting_matchups["batter_dismissals"]
    / batting_matchups["balls_faced"].replace(0, np.nan)
)


# --------------------------------------------------
# 4. Filter to matchups with minimum evidence
# --------------------------------------------------

MIN_MATCHUP_BALLS = 12
MIN_MATCHUP_MATCHES = 2

batting_matchup_shortlist = batting_matchups.loc[
    (
        batting_matchups["balls_faced"]
        >= MIN_MATCHUP_BALLS
    )
    & (
        batting_matchups["matches"]
        >= MIN_MATCHUP_MATCHES
    )
].copy()


# --------------------------------------------------
# 5. Display largest qualifying matchups
# --------------------------------------------------

print("QUALIFYING BATTER-BOWLER MATCHUPS")
print(
    "All aggregated matchups:",
    len(batting_matchups),
)
print(
    "Qualifying matchups:",
    len(batting_matchup_shortlist),
)

display(
    batting_matchup_shortlist
    .sort_values(
        ["year", "match_phase", "balls_faced"],
        ascending=[True, True, False],
    )
    [
        [
            "year",
            "match_phase",
            "bat",
            "bowl",
            "matches",
            "balls_faced",
            "runs_scored",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
            "batter_dismissals",
            "dismissal_rate_pct",
        ]
    ]
    .round(2)
    .reset_index(drop=True)
)

QUALIFYING BATTER-BOWLER MATCHUPS
All aggregated matchups: 746
Qualifying matchups: 17


,year,match_phase,bat,bowl,matches,balls_faced,runs_scored,strike_rate,dot_ball_pct,boundary_ball_pct,batter_dismissals,dismissal_rate_pct
0,2023,Death,Rinku Singh,Matheesha Pathirana,2,13,17,130.77,53.85,23.08,1,7.69
1,2023,Death,Rinku Singh,T Natarajan,2,12,21,175.00,25.00,25.00,1,8.33
2,2023,Middle,Rinku Singh,Ravindra Jadeja,2,17,31,182.35,35.29,23.53,0,0.00
3,2023,Middle,Nitish Rana,Mayank Markande,2,13,14,107.69,38.46,7.69,0,0.00
4,2023,Middle,Nitish Rana,Moeen Ali,2,12,19,158.33,16.67,25.00,0,0.00
5,2023,Powerplay,Rahmanullah Gurbaz,Mohammed Shami,2,14,22,157.14,42.86,21.43,1,7.14
6,2023,Powerplay,Nitish Rana,Marco Jansen,2,12,13,108.33,41.67,16.67,0,0.00
7,2023,Powerplay,Nitish Rana,Tushar Deshpande,2,12,12,100.00,66.67,16.67,0,0.00
8,2024,Middle,Venkatesh Iyer,Piyush Chawla,2,13,11,84.62,53.85,7.69,1,7.69
9,2024,Powerplay,Phil Salt,Khaleel Ahmed,2,22,35,159.09,54.55,31.82,0,0.00


In [6]:
# --------------------------------------------------
# 1. Examine the distribution of matchup workloads
# --------------------------------------------------

print("MATCHUP WORKLOAD DISTRIBUTION")

display(
    batting_matchups["balls_faced"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .to_frame("balls_faced")
)


# --------------------------------------------------
# 2. Count matchups at progressively stronger thresholds
# --------------------------------------------------

thresholds = [12, 18, 24, 30, 36, 48, 60]

workload_summary = pd.DataFrame(
    {
        "minimum_balls": thresholds,
        "qualifying_matchups": [
            int(
                (
                    (batting_matchups["balls_faced"] >= threshold)
                    & (batting_matchups["matches"] >= 2)
                ).sum()
            )
            for threshold in thresholds
        ],
    }
)

display(workload_summary)


# --------------------------------------------------
# 3. Inspect the highest-workload matchups
# --------------------------------------------------

print("TOP 25 MATCHUPS BY LEGAL BALLS FACED")

display(
    batting_matchups
    .sort_values(
        ["balls_faced", "matches"],
        ascending=[False, False],
    )
    [
        [
            "year",
            "match_phase",
            "bat",
            "bowl",
            "matches",
            "balls_faced",
            "runs_scored",
            "strike_rate",
            "dot_ball_pct",
            "boundary_ball_pct",
            "batter_dismissals",
        ]
    ]
    .head(25)
    .round(2)
    .reset_index(drop=True)
)

MATCHUP WORKLOAD DISTRIBUTION


,balls_faced
count,746.000000
mean,4.241287
std,2.990941
min,1.000000
25%,2.000000
50%,4.000000
75%,5.000000
90%,8.000000
95%,10.000000
max,22.000000


,minimum_balls,qualifying_matchups
0,12,17
1,18,1
2,24,0
3,30,0
4,36,0
5,48,0
6,60,0


TOP 25 MATCHUPS BY LEGAL BALLS FACED


,year,match_phase,bat,bowl,matches,balls_faced,runs_scored,strike_rate,dot_ball_pct,boundary_ball_pct,batter_dismissals
0,2024,Powerplay,Phil Salt,Khaleel Ahmed,2,22,35,159.09,54.55,31.82,0
1,2023,Middle,Rinku Singh,Ravindra Jadeja,2,17,31,182.35,35.29,23.53,0
2,2024,Middle,Sunil Narine,Ravichandran Ashwin,1,17,34,200.00,35.29,35.29,0
3,2024,Powerplay,Sunil Narine,Mohammed Siraj,2,16,20,125.00,50.00,18.75,0
4,2024,Powerplay,Rahmanullah Gurbaz,Bhuvneshwar Kumar,2,16,22,137.50,56.25,25.00,0
5,2024,Middle,Shreyas Iyer,Karn Sharma,1,15,19,126.67,40.00,20.00,0
6,2024,Powerplay,Venkatesh Iyer,Bhuvneshwar Kumar,3,14,30,214.29,35.71,35.71,0
7,2023,Powerplay,Rahmanullah Gurbaz,Mohammed Shami,2,14,22,157.14,42.86,21.43,1
8,2023,Death,Rinku Singh,Matheesha Pathirana,2,13,17,130.77,53.85,23.08,1
9,2023,Middle,Nitish Rana,Mayank Markande,2,13,14,107.69,38.46,7.69,0


In [8]:
# --------------------------------------------------
# KKR batting: opponent x season x phase
# --------------------------------------------------

kkr_batting_opponent_phase = kkr_batting_deliveries.copy()

# Use the dataset's batting ball-faced definition.
kkr_batting_opponent_phase["batter_ball_faced"] = (
    kkr_batting_opponent_phase["ballfaced"].eq(1)
)

kkr_batting_opponent_phase["batter_dot_ball"] = (
    kkr_batting_opponent_phase["batter_ball_faced"]
    & kkr_batting_opponent_phase["batruns"].eq(0)
)

kkr_batting_opponent_phase["batter_boundary_ball"] = (
    kkr_batting_opponent_phase["batter_ball_faced"]
    & kkr_batting_opponent_phase["batruns"].isin([4, 6])
)

opponent_batting_summary = (
    kkr_batting_opponent_phase.groupby(
        ["year", "team_bowl", "match_phase"],
        dropna=False,
    )
    .agg(
        matches=("p_match", "nunique"),
        delivery_events=("ball_id", "size"),
        balls_faced=("batter_ball_faced", "sum"),
        legal_balls=("legal_delivery_matchup", "sum"),
        runs_off_bat=("batruns", "sum"),
        dot_balls=("batter_dot_ball", "sum"),
        boundary_balls=("batter_boundary_ball", "sum"),
    )
    .reset_index()
)

opponent_batting_summary["strike_rate"] = (
    100
    * opponent_batting_summary["runs_off_bat"]
    / opponent_batting_summary["balls_faced"].replace(0, np.nan)
)

opponent_batting_summary["dot_ball_pct"] = (
    100
    * opponent_batting_summary["dot_balls"]
    / opponent_batting_summary["balls_faced"].replace(0, np.nan)
)

opponent_batting_summary["boundary_ball_pct"] = (
    100
    * opponent_batting_summary["boundary_balls"]
    / opponent_batting_summary["balls_faced"].replace(0, np.nan)
)

# This is runs off the bat per six legal deliveries,
# not total team scoring rate including extras.
opponent_batting_summary["bat_runs_per_six_legal_balls"] = (
    6
    * opponent_batting_summary["runs_off_bat"]
    / opponent_batting_summary["legal_balls"].replace(0, np.nan)
)

print("KKR BATTING BY OPPONENT, SEASON AND PHASE")

display(
    opponent_batting_summary
    .sort_values(
        ["year", "match_phase", "bat_runs_per_six_legal_balls"],
        ascending=[True, True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

KKR BATTING BY OPPONENT, SEASON AND PHASE


,year,team_bowl,match_phase,matches,delivery_events,balls_faced,legal_balls,runs_off_bat,dot_balls,boundary_balls,strike_rate,dot_ball_pct,boundary_ball_pct,bat_runs_per_six_legal_balls
0,2023,Royal Challengers Bengaluru,Death,2,64,61,60,125,19,21,204.92,31.15,34.43,12.50
1,2023,Punjab Kings,Death,2,37,36,36,69,8,9,191.67,22.22,25.00,11.50
2,2023,Lucknow Super Giants,Death,1,33,30,30,57,12,9,190.00,40.00,30.00,11.40
3,2023,Gujarat Titans,Death,2,62,60,60,101,22,14,168.33,36.67,23.33,10.10
4,2023,Sunrisers Hyderabad,Death,2,65,61,60,100,18,13,163.93,29.51,21.31,10.00
5,2023,Mumbai Indians,Death,1,33,30,30,42,12,7,140.00,40.00,23.33,8.40
6,2023,Chennai Super Kings,Death,2,59,52,51,67,21,8,128.85,40.38,15.38,7.88
7,2023,Rajasthan Royals,Death,1,30,30,30,33,13,3,110.00,43.33,10.00,6.60
8,2023,Delhi Capitals,Death,1,35,30,30,28,21,4,93.33,70.00,13.33,5.60
9,2023,Gujarat Titans,Middle,2,111,108,108,176,35,24,162.96,32.41,22.22,9.78


In [10]:
# --------------------------------------------------
# KKR bowling: opponent x season x phase
# --------------------------------------------------

kkr_bowling_opponent_phase = kkr_bowling_deliveries.copy()

kkr_bowling_opponent_phase["bowler_credited_wicket"] = (
    kkr_bowling_opponent_phase["out"].eq(True)
    & kkr_bowling_opponent_phase["dismissal"].isin(
        [
            "caught",
            "bowled",
            "leg before wicket",
            "stumped",
            "hit wicket",
        ]
    )
)

kkr_bowling_opponent_phase["bowling_dot_ball"] = (
    kkr_bowling_opponent_phase["legal_delivery_matchup"]
    & kkr_bowling_opponent_phase["score"].eq(0)
)

kkr_bowling_opponent_phase["boundary_ball"] = (
    kkr_bowling_opponent_phase["legal_delivery_matchup"]
    & kkr_bowling_opponent_phase["ballfaced"].eq(1)
    & kkr_bowling_opponent_phase["batruns"].isin([4, 6])
)

opponent_bowling_summary = (
    kkr_bowling_opponent_phase.groupby(
        ["year", "team_bat", "match_phase"],
        dropna=False,
    )
    .agg(
        matches=("p_match", "nunique"),
        delivery_events=("ball_id", "size"),
        legal_balls=("legal_delivery_matchup", "sum"),
        runs_conceded=("score", "sum"),
        dot_balls=("bowling_dot_ball", "sum"),
        bowler_credited_wickets=("bowler_credited_wicket", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        wides=("wide", "sum"),
        no_ball_extras=("noball", "sum"),
    )
    .reset_index()
)

opponent_bowling_summary["overs_bowled"] = (
    opponent_bowling_summary["legal_balls"] / 6
)

opponent_bowling_summary["economy"] = (
    opponent_bowling_summary["runs_conceded"]
    / opponent_bowling_summary["overs_bowled"].replace(0, np.nan)
)

opponent_bowling_summary["dot_ball_pct"] = (
    100
    * opponent_bowling_summary["dot_balls"]
    / opponent_bowling_summary["legal_balls"].replace(0, np.nan)
)

opponent_bowling_summary["wickets_per_20_overs"] = (
    20
    * opponent_bowling_summary["bowler_credited_wickets"]
    / opponent_bowling_summary["overs_bowled"].replace(0, np.nan)
)

opponent_bowling_summary["boundary_balls_per_100_legal"] = (
    100
    * opponent_bowling_summary["boundary_balls"]
    / opponent_bowling_summary["legal_balls"].replace(0, np.nan)
)

opponent_bowling_summary["extras_per_over"] = (
    opponent_bowling_summary["wides"].fillna(0)
    + opponent_bowling_summary["no_ball_extras"].fillna(0)
) / opponent_bowling_summary["overs_bowled"].replace(0, np.nan)

print("OPPONENT SCORING AGAINST KKR BY SEASON AND PHASE")

display(
    opponent_bowling_summary
    .sort_values(
        ["year", "match_phase", "economy"],
        ascending=[True, True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

OPPONENT SCORING AGAINST KKR BY SEASON AND PHASE


,year,team_bat,match_phase,matches,delivery_events,legal_balls,runs_conceded,dot_balls,bowler_credited_wickets,boundary_balls,wides,no_ball_extras,overs_bowled,economy,dot_ball_pct,wickets_per_20_overs,boundary_balls_per_100_legal,extras_per_over
0,2023,Gujarat Titans,Death,2,50,47,123,3,1,14,2,1,7.83,15.70,6.38,2.55,29.79,0.38
1,2023,Chennai Super Kings,Death,2,68,60,127,15,3,14,6,3,10.00,12.70,25.00,6.00,23.33,0.90
2,2023,Lucknow Super Giants,Death,1,35,30,57,8,3,7,5,0,5.00,11.40,26.67,12.00,23.33,1.00
3,2023,Punjab Kings,Death,2,63,60,103,18,3,13,3,0,10.00,10.30,30.00,6.00,21.67,0.30
4,2023,Sunrisers Hyderabad,Death,2,65,60,103,15,4,13,4,1,10.00,10.30,25.00,8.00,21.67,0.50
5,2023,Mumbai Indians,Death,1,17,16,23,3,2,3,1,0,2.67,8.62,18.75,15.00,18.75,0.38
6,2023,Royal Challengers Bengaluru,Death,2,47,46,62,15,3,6,2,0,7.67,8.09,32.61,7.83,13.04,0.26
7,2023,Delhi Capitals,Death,1,29,26,24,9,2,0,2,1,4.33,5.54,34.62,9.23,0.00,0.69
8,2023,Rajasthan Royals,Middle,1,43,43,73,13,0,10,0,0,7.17,10.19,30.23,0.00,23.26,0.00
9,2023,Mumbai Indians,Middle,1,55,54,91,21,2,13,1,0,9.00,10.11,38.89,4.44,24.07,0.11


1. Tactical Findings
Batting matchups
- KKR recorded strong powerplay scoring against RCB and DC in 2024, with strike rates above 200 in both matchups.
- KKR also produced high death-over scoring rates against SRH, PBKS and DC in 2024.
- Middle-over results varied by opponent. For example, KKR scored 216 runs off the bat against SRH but 41 against CSK in the 2024 middle phase.
- These results identify matchups worth investigating further, but they do not establish persistent opponent-specific weaknesses.
Bowling matchups
- KKR's bowling performance varied across opponents and match phases.
- In the 2024 middle phase, KKR recorded 10 credited wickets against LSG at an economy of 7.72 runs per over across two matches.
- Against DC in the same phase, KKR recorded eight credited wickets at an economy of 8.28 across two matches.
- Death bowling remained a mixed picture. KKR took seven credited wickets against SRH across three matches at an economy of 10.23, but some individual matchups were considerably more expensive.
Analytical interpretation
The findings suggest that tactical analysis should consider wicket-taking, run prevention and scoring opportunities together, rather than relying on a single metric.
These are descriptive observations, not causal explanations or reliable predictions of future performance.
2. Limitations
1. Small samples: Most opponent-phase combinations cover only one to three matches.
2. Player matchup limitations: Batter-versus-bowler combinations generally have very few balls, preventing robust individual matchup recommendations.
3. Context is not fully controlled: Venue, pitch conditions, batting order, target, wickets in hand and match situation can influence the observed results.
4. Season coverage differs: The supplied dataset omits KKR's final two league matches in 2024 while retaining the Qualifier 1 and Final. Season comparisons must acknowledge this difference.
5. Metric definitions matter: Batting balls faced and legal deliveries are different denominators. Bowling economy must use legal deliveries, and boundary-event rates must clearly specify how illegal deliveries are handled.
6. Association is not causation: The analysis cannot establish that a particular player or tactical decision caused a match outcome.
3. Conclusion
The tactical analysis provides exploratory evidence for further investigation, rather than definitive player-specific instructions. Its primary value is to identify potentially important phase-and-opponent patterns while making the limitations of the available data explicit.